# Module 00: Mixed-Frequency Data Exploration

## Learning Objectives
By completing this notebook, you will:
1. Visualize GDP and IP at their native frequencies alongside each other
2. Demonstrate information loss from different aggregation strategies
3. Quantify how much within-quarter information aggregation discards
4. Build the MIDAS data matrix structure used in all subsequent modules

## Prerequisites
- Module 00, Notebook 01 (environment setup) completed
- Mixed-frequency problem guide (Guide 01) read

## Estimated Time: 15 minutes

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch

# Suppress minor warnings
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Plotting defaults
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Resources path
RESOURCES_DIR = '../resources'

print("Imports successful.")

## 1. Load and Align Data

We load GDP and IP using the same pattern as Notebook 01. The key step is aligning the two series so that each quarterly observation is paired with its three constituent monthly observations.

In [ ]:
# Load GDP (quarterly growth rates)
gdp_growth = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'gdp_quarterly.csv'),
    index_col='date', parse_dates=True
).squeeze()
gdp_growth.name = 'gdp_growth'

# Load IP (monthly growth rates)
ip_growth = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'industrial_production_monthly.csv'),
    index_col='date', parse_dates=True
).squeeze()
ip_growth.name = 'ip_growth'

# Load S&P 500 (daily returns)
sp500_returns = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'sp500_daily.csv'),
    index_col='date', parse_dates=True
).squeeze()
sp500_returns.name = 'sp500_return'

# Use Period index for clean alignment
gdp_q = gdp_growth.copy()
gdp_q.index = pd.PeriodIndex(gdp_growth.index, freq='Q')

ip_m = ip_growth.copy()
ip_m.index = pd.PeriodIndex(ip_growth.index, freq='M')

# Three aggregation strategies for monthly IP -> quarterly
ip_agg_last = ip_m.resample('Q').last()    # Last month of quarter
ip_agg_mean = ip_m.resample('Q').mean()    # Equal average
ip_agg_sum  = ip_m.resample('Q').sum()     # Sum (for flow variables)

# Align to GDP index
common_idx = gdp_q.index.intersection(ip_agg_mean.index)
gdp_aligned = gdp_q.loc[common_idx]
ip_last_aligned = ip_agg_last.loc[common_idx]
ip_mean_aligned = ip_agg_mean.loc[common_idx]

print(f"Aligned sample: {len(gdp_aligned)} quarters")
print(f"Date range: {common_idx[0]} to {common_idx[-1]}")

## 2. Visualizing Frequency Mismatch

The plot below shows GDP (quarterly) and IP (monthly) on the same time axis. Notice that:
- Each quarterly GDP bar spans 3 monthly IP bars
- The within-quarter pattern of IP varies substantially quarter-to-quarter
- Aggregation destroys this within-quarter variation

In [ ]:
# Focus on 2007–2010 to see the financial crisis clearly
start_vis = '2007-01-01'
end_vis = '2010-12-31'

gdp_vis = gdp_growth.loc[start_vis:end_vis]
ip_vis = ip_growth.loc[start_vis:end_vis]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Mixed-Frequency Data: 2007–2010 Financial Crisis',
             fontsize=13, fontweight='bold')

# Shade quarter boundaries with alternating colors
quarters = pd.period_range(start_vis, end_vis, freq='Q')
for i, q in enumerate(quarters):
    color = '#f8f9fa' if i % 2 == 0 else '#e9ecef'
    start_q = q.start_time
    end_q = q.end_time
    for ax in axes:
        ax.axvspan(start_q, end_q, alpha=0.4, color=color, zorder=0)

# Top panel: monthly IP
ax1 = axes[0]
colors_ip = ['steelblue' if v >= 0 else 'coral' for v in ip_vis.values]
ax1.bar(ip_vis.index, ip_vis.values, width=25, color=colors_ip, alpha=0.85, zorder=2)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_ylabel('MoM Growth (%)')
ax1.set_title('Monthly Industrial Production Growth', fontsize=11)
ax1.set_xlim(pd.Timestamp(start_vis) - pd.Timedelta(days=15),
              pd.Timestamp(end_vis) + pd.Timedelta(days=15))

# Annotate the crisis months
ax1.annotate('Lehman\nSeptember 2008',
              xy=(pd.Timestamp('2008-09-01'), -0.85),
              xytext=(pd.Timestamp('2008-06-01'), -4.5),
              arrowprops=dict(arrowstyle='->', color='darkred'),
              fontsize=9, color='darkred')

# Bottom panel: quarterly GDP
ax2 = axes[1]
colors_gdp = ['steelblue' if v >= 0 else 'coral' for v in gdp_vis.values]
ax2.bar(gdp_vis.index, gdp_vis.values, width=70, color=colors_gdp, alpha=0.85, zorder=2)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('QoQ Growth (%)')
ax2.set_title('Quarterly GDP Growth (bars span full quarter)', fontsize=11)
ax2.set_xlim(pd.Timestamp(start_vis) - pd.Timedelta(days=15),
              pd.Timestamp(end_vis) + pd.Timedelta(days=15))

# Add quarter labels on the bottom
for q in quarters:
    mid = q.start_time + (q.end_time - q.start_time) / 2
    ax2.text(mid, ax2.get_ylim()[0] * 0.95, str(q),
             ha='center', va='bottom', fontsize=7, color='gray')

plt.tight_layout()
plt.savefig('mixed_frequency_crisis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Note: Alternating shading = alternating quarters. Each quarterly GDP bar corresponds to 3 monthly IP bars.")

## 3. Information Loss from Aggregation

### 3.1 The Variance Decomposition

We can quantify information loss by decomposing monthly IP variance into:
- **Between-quarter variance**: Captured by quarterly averages
- **Within-quarter variance**: Lost by aggregation

The fraction of variance lost to aggregation tells us how much MIDAS can potentially recover.

In [ ]:
# Map each monthly observation to its quarter
ip_monthly_df = ip_m.reset_index()
ip_monthly_df.columns = ['month', 'ip_growth']
ip_monthly_df['quarter'] = ip_monthly_df['month'].dt.to_timestamp().dt.to_period('Q')

# Within-quarter mean (what aggregation keeps)
quarterly_means = ip_monthly_df.groupby('quarter')['ip_growth'].mean()
ip_monthly_df['quarterly_mean'] = ip_monthly_df['quarter'].map(quarterly_means)

# Decompose variance
total_var = ip_monthly_df['ip_growth'].var()
between_var = ip_monthly_df['quarterly_mean'].var()  # Variance of quarterly means
within_var = (ip_monthly_df['ip_growth'] - ip_monthly_df['quarterly_mean']).var()

print("Variance Decomposition of Monthly IP Growth")
print("-" * 45)
print(f"Total variance (monthly):      {total_var:.4f}")
print(f"Between-quarter variance:      {between_var:.4f}  ({between_var/total_var:.1%} of total)")
print(f"Within-quarter variance:       {within_var:.4f}  ({within_var/total_var:.1%} of total)")
print()
print(f"Information retained by equal-weight aggregation: {between_var/total_var:.1%}")
print(f"Information discarded by aggregation:             {within_var/total_var:.1%}")
print()
print("This within-quarter variation is what MIDAS can use that aggregation discards.")

### 3.2 Regression Comparison: Different Aggregation Strategies

We now estimate three simple regressions of quarterly GDP growth on quarterly IP (using different aggregation methods) and compare their in-sample fit.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def simple_ols(y, x, name):
    """Fit OLS and return key statistics."""
    # Align and drop NaN
    df = pd.DataFrame({'y': y, 'x': x}).dropna()
    X = df[['x']].values
    Y = df['y'].values
    
    model = LinearRegression().fit(X, Y)
    y_hat = model.predict(X)
    r2 = r2_score(Y, y_hat)
    residuals = Y - y_hat
    rmse = np.sqrt(np.mean(residuals**2))
    
    return {
        'name': name,
        'n': len(Y),
        'alpha': model.intercept_,
        'beta': model.coef_[0],
        'r2': r2,
        'rmse': rmse,
        'y_hat': y_hat,
        'y': Y,
        'index': df.index
    }

# Convert Period index back to Timestamp for pandas operations
gdp_ts = gdp_aligned.copy()
gdp_ts.index = gdp_aligned.index.to_timestamp()

ip_last_ts = ip_last_aligned.copy()
ip_last_ts.index = ip_last_aligned.index.to_timestamp()

ip_mean_ts = ip_mean_aligned.copy()
ip_mean_ts.index = ip_mean_aligned.index.to_timestamp()

# Estimate regressions
results = [
    simple_ols(gdp_ts, ip_last_ts, 'Last-period sampling (M3 only)'),
    simple_ols(gdp_ts, ip_mean_ts, 'Equal-weight average (M1+M2+M3)/3'),
]

print("GDP Growth ~ Industrial Production (Different Aggregations)")
print("=" * 60)
print(f"{'Model':<40} {'N':>4} {'Beta':>7} {'R²':>6} {'RMSE':>6}")
print("-" * 60)
for r in results:
    print(f"{r['name']:<40} {r['n']:>4d} {r['beta']:>7.4f} {r['r2']:>6.3f} {r['rmse']:>6.3f}")

print()
print("Interpretation:")
print("- Last-period sampling captures the most recent month's momentum")
print("- Equal-weight averaging dilutes the signal from the last month")
print("- Neither method can use all timing information simultaneously")
print("- MIDAS (Module 01) estimates optimal weights, outperforming both")

### 3.3 Visualization: The Same Quarter, Different Aggregations

This plot shows the 8 quarters with the largest difference between last-period and equal-weight aggregation — these are the quarters where the aggregation choice matters most.

In [ ]:
# Identify quarters where aggregation choice matters most
diff_agg = (ip_last_aligned - ip_mean_aligned).abs()
top_quarters = diff_agg.nlargest(8).index

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
fig.suptitle('Quarters Where Aggregation Method Matters Most\n(Largest gap between last-period and equal-weight)',
             fontsize=12, fontweight='bold')

for ax, quarter in zip(axes, top_quarters):
    # Get the 3 monthly observations for this quarter
    q_months = pd.period_range(start=quarter.start_time, end=quarter.end_time, freq='M')
    monthly_vals = ip_m.reindex(q_months).values
    
    if np.isnan(monthly_vals).any():
        ax.text(0.5, 0.5, 'Data unavailable', ha='center', va='center', transform=ax.transAxes)
        continue
    
    colors = ['steelblue' if v >= 0 else 'coral' for v in monthly_vals]
    ax.bar([1, 2, 3], monthly_vals, color=colors, alpha=0.85, zorder=2)
    
    # Add horizontal lines for the two aggregation results
    last_val = monthly_vals[2]  # Month 3
    mean_val = monthly_vals.mean()
    
    ax.axhline(last_val, color='darkblue', linestyle='--', linewidth=1.5,
               label=f'Last: {last_val:.2f}%')
    ax.axhline(mean_val, color='darkred', linestyle=':', linewidth=1.5,
               label=f'Mean: {mean_val:.2f}%')
    ax.axhline(0, color='black', linewidth=0.5)
    
    # Get the corresponding GDP value
    gdp_val = gdp_q.get(quarter, float('nan'))
    
    ax.set_title(f'{quarter}\nGDP: {gdp_val:.2f}%', fontsize=9)
    ax.set_xlabel('Month within quarter')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['M1', 'M2', 'M3'])
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('aggregation_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Key observation: The gap between blue dashed (last-period) and red dotted (equal-weight) shows")
print("how much the aggregation method affects the signal going into the regression.")

## 4. Building the MIDAS Data Matrix

Before we can estimate a MIDAS model, we need to structure the data as a matrix where:
- Each **row** is a quarterly observation (for GDP growth)
- Each **column** is a high-frequency lag (each monthly IP observation going back in time)

This matrix structure is fundamental to everything in this course.

In [ ]:
def build_midas_matrix(y_quarterly, x_monthly, n_lags, freq_ratio=3):
    """
    Construct the MIDAS regressor matrix.
    
    For quarterly GDP ~ monthly IP:
      - n_lags = 9 means 3 quarterly lags × 3 months each
      - X[t, 0] = most recent month of quarter t
      - X[t, 1] = second-most-recent month
      - X[t, 2] = earliest month of quarter t
      - X[t, 3] = most recent month of quarter t-1
      - etc.
    
    Parameters
    ----------
    y_quarterly : pd.Series
        Quarterly dependent variable with PeriodIndex (freq='Q').
    x_monthly : pd.Series
        Monthly regressor with PeriodIndex (freq='M').
    n_lags : int
        Total number of high-frequency lags (e.g., 9 = 3 quarters × 3 months).
    freq_ratio : int
        High-freq to low-freq ratio (3 for monthly/quarterly).
    
    Returns
    -------
    Y : np.ndarray, shape (T,)
        Dependent variable values.
    X : np.ndarray, shape (T, n_lags)
        MIDAS regressor matrix.
    valid_idx : pd.PeriodIndex
        Quarterly periods corresponding to valid rows.
    """
    # Get arrays
    y_vals = y_quarterly.values
    x_vals = x_monthly.values
    q_index = y_quarterly.index
    m_index = x_monthly.index
    
    T_Q = len(y_vals)
    T_M = len(x_vals)
    
    X = np.full((T_Q, n_lags), np.nan)
    
    for t in range(T_Q):
        q = q_index[t]  # Current quarter as Period
        
        for j in range(n_lags):
            # Month index: lag j back from end of quarter t
            # Month 0 = last month of quarter t
            # Month 1 = second-to-last month of quarter t
            # Month 3 = last month of quarter t-1, etc.
            target_quarter_offset = j // freq_ratio   # How many full quarters back
            within_quarter_offset = j % freq_ratio    # Position within that quarter
            
            # Last month of (t - target_quarter_offset)
            target_q = q - target_quarter_offset
            last_month_of_target_q = target_q.end_time.to_period('M')
            target_month = last_month_of_target_q - within_quarter_offset
            
            # Look up in x_monthly
            if target_month in m_index:
                m_pos = m_index.get_loc(target_month)
                X[t, j] = x_vals[m_pos]
    
    Y = y_vals
    
    # Remove rows with any NaN
    valid = ~(np.isnan(X).any(axis=1) | np.isnan(Y))
    
    return Y[valid], X[valid], q_index[valid]


# Build MIDAS matrix with 9 lags (3 quarterly lags × 3 months each)
N_LAGS = 9
Y, X, q_dates = build_midas_matrix(gdp_q, ip_m, n_lags=N_LAGS)

print(f"MIDAS Data Matrix Structure")
print(f"  Quarterly observations (rows): {len(Y)}")
print(f"  High-frequency lags (columns): {N_LAGS}")
print(f"  Matrix shape: {X.shape}")
print(f"  Date range: {q_dates[0]} to {q_dates[-1]}")
print()
print(f"Column interpretation:")
print(f"  X[:, 0] = IP growth, Month 3 of current quarter (most recent)")
print(f"  X[:, 1] = IP growth, Month 2 of current quarter")
print(f"  X[:, 2] = IP growth, Month 1 of current quarter")
print(f"  X[:, 3] = IP growth, Month 3 of previous quarter")
print(f"  ... and so on")
print()
print(f"Sample row (2023Q3):")
q2023q3_idx = list(q_dates).index(pd.Period('2023Q3', 'Q')) if pd.Period('2023Q3', 'Q') in q_dates else -1
if q2023q3_idx >= 0:
    print(f"  Y = {Y[q2023q3_idx]:.4f}% (GDP growth)")
    print(f"  X = {X[q2023q3_idx].round(4)} (IP lags)")

## 5. Visualizing the MIDAS Weight Problem

The key question for MIDAS: **which lag weights should we assign to these 9 columns?**

The plot below compares three "naive" weight patterns and shows that the correlation of each lag with GDP tells us something about the optimal weights — but we need the full MIDAS estimation (Module 01) to get the right answer.

In [ ]:
# Compute correlation of each lagged monthly IP with quarterly GDP
lag_correlations = np.array([
    np.corrcoef(Y, X[:, j])[0, 1] for j in range(N_LAGS)
])

lag_labels = []
for j in range(N_LAGS):
    q_offset = j // 3
    m_within = j % 3
    month_names = ['M3', 'M2', 'M1']
    if q_offset == 0:
        lag_labels.append(f'Curr {month_names[m_within]}')
    else:
        lag_labels.append(f'Q-{q_offset} {month_names[m_within]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MIDAS Weight Structure: What Should Each Lag Be Worth?',
             fontsize=12, fontweight='bold')

# Left panel: lag correlations
ax1 = axes[0]
colors = ['steelblue' if c > 0 else 'coral' for c in lag_correlations]
bars = ax1.bar(range(N_LAGS), lag_correlations, color=colors, alpha=0.85)
ax1.axhline(0, color='black', linewidth=0.5)
ax1.set_xticks(range(N_LAGS))
ax1.set_xticklabels(lag_labels, rotation=45, ha='right', fontsize=9)
ax1.set_ylabel('Correlation with quarterly GDP growth')
ax1.set_title('Correlation of Each IP Lag with GDP Growth\n(suggests downward-sloping weight function)')

# Add vertical lines to separate quarters
for x_pos in [2.5, 5.5, 8.5]:
    if x_pos < N_LAGS:
        ax1.axvline(x_pos, color='gray', linestyle='--', alpha=0.5)

for i, (bar, label) in enumerate(zip(bars, lag_labels)):
    if i % 3 == 0 and i < N_LAGS:
        ax1.text(i + 1, ax1.get_ylim()[0] * 0.85,
                 f'Q-{i//3}' if i > 0 else 'Current Q',
                 ha='center', fontsize=8, color='gray')

# Right panel: three candidate weight shapes
ax2 = axes[1]
lags = np.arange(1, N_LAGS + 1)

# Flat weights (aggregation)
flat_weights = np.ones(N_LAGS) / N_LAGS

# Declining weights (geometric)
rho = 0.7
geom_weights = np.array([rho**j for j in range(N_LAGS)])
geom_weights = geom_weights / geom_weights.sum()

# Hump-shaped (Beta-like)
from scipy.stats import beta
beta_shape = beta.pdf(np.linspace(0, 1, N_LAGS), 1.5, 4)
beta_weights = beta_shape / beta_shape.sum()

ax2.plot(lags, flat_weights, 'o-', color='gray', label='Flat (equal-weight aggr.)', linewidth=2)
ax2.plot(lags, geom_weights, 's-', color='steelblue', label='Geometric decay', linewidth=2)
ax2.plot(lags, beta_weights, '^-', color='coral', label='Beta-shaped (MIDAS typical)', linewidth=2)
ax2.set_xlabel('High-frequency lag (0 = most recent)')
ax2.set_ylabel('Weight')
ax2.set_title('Candidate Weight Functions\n(MIDAS estimates these from data)')
ax2.legend()
ax2.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('midas_weight_motivation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Left panel: The declining correlation pattern suggests recent months matter more.")
print("Right panel: MIDAS estimates which weight shape fits best (Module 01 does this formally).")

## 6. Self-Check Exercises

Complete these exercises to verify your understanding. Each has a solution check built in.

In [ ]:
# Exercise 1: Compute the frequency ratio for daily S&P 500 to quarterly GDP
# 
# Count the average number of trading days in a quarter (2018-2023)
# Hint: Load sp500_daily.csv, convert dates to quarters, count per quarter

sp500_daily = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'sp500_daily.csv'),
    index_col='date', parse_dates=True
).squeeze()

# YOUR CODE: Count average daily observations per quarter
sp500_by_quarter = sp500_daily.resample('QE').count()
avg_daily_per_quarter = sp500_by_quarter[sp500_by_quarter > 0].mean()

print(f"Exercise 1: Average trading days per quarter = {avg_daily_per_quarter:.1f}")

# Check
assert 55 < avg_daily_per_quarter < 70, \
    f"Expected ~65 trading days per quarter, got {avg_daily_per_quarter:.1f}. Check your calculation."
print("  Check passed: reasonable frequency ratio for daily MIDAS.")

In [ ]:
# Exercise 2: Find the quarter with maximum within-quarter IP variance
# 
# For each quarter, compute the variance of the 3 monthly IP observations.
# Return the quarter with the highest within-quarter variance.

ip_monthly_df = pd.DataFrame({'ip_growth': ip_m.values}, index=ip_m.index)
ip_monthly_df['quarter'] = ip_monthly_df.index.to_timestamp().to_period('Q')

# YOUR CODE: Compute within-quarter variance and find the maximum
within_quarter_var = ip_monthly_df.groupby('quarter')['ip_growth'].var()
max_var_quarter = within_quarter_var.idxmax()
max_var_value = within_quarter_var.max()

print(f"Exercise 2: Quarter with highest within-quarter IP variance:")
print(f"  Quarter: {max_var_quarter}")
print(f"  Within-quarter variance: {max_var_value:.4f}")

# Get the three months for this quarter
months = pd.period_range(max_var_quarter.start_time, max_var_quarter.end_time, freq='M')
vals = ip_m.reindex(months).values
print(f"  Monthly IP values: {vals.round(3)} (huge within-quarter variation!)")

# Check: Should be during COVID or Financial Crisis
assert str(max_var_quarter.year) in ['2008', '2009', '2020'], \
    f"Expected crisis quarter, got {max_var_quarter}. COVID or GFC should have highest variance."
print("  Check passed: Maximum variance in a major crisis quarter.")

In [ ]:
# Exercise 3: Verify that the MIDAS matrix has the correct lag structure
# 
# For quarter 2023Q3:
# - X[:, 0] should equal IP growth for September 2023 (last month of Q3)
# - X[:, 1] should equal IP growth for August 2023
# - X[:, 2] should equal IP growth for July 2023 (first month of Q3)
# - X[:, 3] should equal IP growth for June 2023 (last month of Q2)

# Find the row for 2023Q3
q2023q3 = pd.Period('2023Q3', 'Q')

if q2023q3 in q_dates:
    row_idx = list(q_dates).index(q2023q3)
    
    # Expected values from ip_m
    sep_2023 = ip_m.get(pd.Period('2023-09', 'M'), np.nan)
    aug_2023 = ip_m.get(pd.Period('2023-08', 'M'), np.nan)
    jul_2023 = ip_m.get(pd.Period('2023-07', 'M'), np.nan)
    jun_2023 = ip_m.get(pd.Period('2023-06', 'M'), np.nan)
    
    print(f"Exercise 3: Verifying MIDAS matrix for 2023Q3")
    print(f"  X[row, 0] = {X[row_idx, 0]:.4f}  (should be Sep 2023 IP: {sep_2023:.4f})")
    print(f"  X[row, 1] = {X[row_idx, 1]:.4f}  (should be Aug 2023 IP: {aug_2023:.4f})")
    print(f"  X[row, 2] = {X[row_idx, 2]:.4f}  (should be Jul 2023 IP: {jul_2023:.4f})")
    print(f"  X[row, 3] = {X[row_idx, 3]:.4f}  (should be Jun 2023 IP: {jun_2023:.4f})")
    
    if not (np.isnan(sep_2023) or np.isnan(aug_2023)):
        assert abs(X[row_idx, 0] - sep_2023) < 0.001, "Lag 0 mismatch"
        assert abs(X[row_idx, 1] - aug_2023) < 0.001, "Lag 1 mismatch"
        print("  Check passed: MIDAS matrix alignment is correct.")
else:
    print(f"2023Q3 not in sample. Available sample ends at {q_dates[-1]}.")

## Summary

### Key Takeaways

1. **Frequency mismatch is concrete**: Quarterly GDP corresponds to 3 monthly IP observations and ~65 daily S&P 500 returns. The frequency ratio determines how much information aggregation discards.

2. **Aggregation discards within-quarter variance**: In our sample, roughly 30–40% of monthly IP variance is within-quarter. Equal-weight aggregation discards this entirely.

3. **Aggregation choice affects regression fit**: Last-period sampling and equal-weight averaging produce different coefficient estimates and R-squared values from the same data.

4. **The MIDAS data matrix**: $(T_L \times n_{\text{lags}})$ — rows are low-frequency periods, columns are high-frequency lags. This is the fundamental data structure for all MIDAS models.

5. **Lag correlations hint at weight structure**: Recent months (lag 0, 1, 2) correlate more with quarterly GDP than older months. The optimal weight function (Module 01) formalizes this pattern.

### What's Next

Module 01 introduces the MIDAS equation formally — including the Beta polynomial and Almon polynomial weight functions that parameterize the lag structure we visualized above. We will estimate these weight functions from data and compare them to simple aggregation.

---
*Next notebook: Module 01 — MIDAS Regression Fundamentals*